In [1]:
import numpy as np
import pandas as pd
import torch
import torchvision
import matplotlib.pyplot as plt
import os
import argparse
import matplotlib
from collections import OrderedDict
from datetime import datetime
from PIL import Image
from models_utils import * 
from data_utils import *
from tqdm import tqdm

F:\anaconda\envs\pytorch\lib\site-packages\torchvision\datasets\mnist.py:498: UserWarning: The given NumPy array is not writeable, and PyTorch does not support non-writeable tensors. This means you can write to the underlying (supposedly non-writeable) NumPy array using the tensor. You may want to copy the array to protect its data or make it writeable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at  ..\torch\csrc\utils\tensor_numpy.cpp:180.)
  return torch.from_numpy(parsed.astype(m[2], copy=False)).view(*s)


In [2]:
S1 = ['KMNIST','FMNIST','MNIST','KaMNIST']
S2 = ['KMNIST','FMNIST','KaMNIST','MNIST']
S3 = ['KMNIST','MNIST','FMNIST','KaMNIST']
S4 = ['KMNIST','MNIST','KaMNIST','FMNIST']
S5 = ['KMNIST','KaMNIST','MNIST','FMNIST']
S6 = ['KMNIST','KaMNIST','FMNIST','MNIST']

S7 = ['FMNIST','KMNIST','MNIST','KaMNIST']
S8 = ['FMNIST','KMNIST','KaMNIST','MNIST']
S9 = ['FMNIST','MNIST','KMNIST','KaMNIST']
S10 = ['FMNIST','MNIST','KaMNIST','KMNIST']
S11 = ['FMNIST','KaMNIST','KMNIST','MNIST']
S12 = ['FMNIST','KaMNIST','MNIST','KMNIST']

S13 = ['MNIST','FMNIST','KMNIST','KaMNIST']
S14 = ['MNIST','FMNIST','KaMNIST','KMNIST']
S15 = ['MNIST','KMNIST','FMNIST','KaMNIST']
S16 = ['MNIST','KMNIST','KaMNIST','FMNIST']
S17 = ['MNIST','KaMNIST','FMNIST','KMNIST']
S18 = ['MNIST','KaMNIST','KMNIST','FMNIST']

S19 = ['KaMNIST','FMNIST','KMNIST','MNIST']
S20 = ['KaMNIST','FMNIST','MNIST','KMNIST']
S21 = ['KaMNIST','MNIST','FMNIST','KMNIST']
S22 = ['KaMNIST','MNIST','KMNIST','FMNIST']
S23 = ['KaMNIST','KMNIST','MNIST','FMNIST']
S24 = ['KaMNIST','KMNIST','FMNIST','MNIST']

len([S1,S2,S3,S4,S5,S6,S7,S8,S9,S10,S11,S12,S13,S14,S15,S16,S17,S18,S19,S20,S21,S22,S23,S24])

24

In [3]:
#保证随机种子的一致性
for fit_para in [[59.01,59.40,-63.56,-63.23]]:
    
    class Args():
        def __init__(self,scenario='task',net='bnn',in_size=784,hidden_layers=[1000,500],out_size=10,task_sequence=S1,
                    lr=0.005,gamma=1,epochs_per_task=20,norm='bn',meta=[10],rnd_consolidation=False,ewc_lambda=0,ewc=False,si_lambda=0,si=False,bin_path=False,decay=1e-7,init='uniform',init_width=0.5,
                    save=True,interleaved=False,beaker=False,fb=5e-3,n_bk=4,ratios=[1e-2,1e-3,1e-4,1e-5],areas=[1,2,4,8],
                    device=0,seed=0,bit_num=3,upper_bound=1,noise_std=0.01,Ap=fit_para[0],Bp=fit_para[1],An=fit_para[2],Bn=fit_para[3] ):
            self.scenario=scenario
            self.net=net
            self.in_size=in_size
            self.hidden_layers=hidden_layers
            self.out_size=out_size
            self.task_sequence=task_sequence
            self.lr=lr
            self.gamma=gamma
            self.epochs_per_task=epochs_per_task
            self.norm=norm
            self.meta=meta
            self.rnd_consolidation=rnd_consolidation
            self.ewc_lambda=ewc_lambda
            self.ewc=ewc
            self.si_lambda=si_lambda
            self.si=si
            self.bin_path=bin_path
            self.decay=decay
            self.init=init
            self.init_width=init_width
            self.save=save
            self.interleaved=interleaved
            self.beaker=beaker
            self.fb=fb
            self.n_bk=n_bk
            self.ratios=ratios
            self.areas=areas
            self.device=device
            self.seed=seed
            self.bit_num=bit_num
            self.upper_bound=upper_bound
            self.noise_std=noise_std
            self.Ap=Ap
            self.Bp=Bp
            self.An=An
            self.Bn=Bn
            
            
            
    args=Args()
    #print(args)
    device = torch.device("cuda:"+str(args.device) if torch.cuda.is_available() else "cpu")

    if args.seed is not None:
        torch.manual_seed(args.seed)
        np.random.seed(args.seed)

    date = datetime.now().strftime('%Y-%m-%d')
    time = datetime.now().strftime('%H-%M-%S')
    path = 'results/'+date+'/'+time+'_gpu'+str(args.device)
    ope_path = r'E:\2024-2025-1\research\BNN&CL\AA_neural_network_simulation'
    select_path = ope_path + '\\' + 'dif_conductance_up_limit'

    if not(os.path.exists(path)):
        os.makedirs(path)
        
    if not(os.path.exists(select_path)):
        os.makedirs(select_path)
        
    createHyperparametersFile(path, args)
    train_loader_list = []
    test_loader_list = []
    dset_train_list = []
    task_names = []
    
    for idx, task in enumerate(args.task_sequence):
        if task == 'MNIST':
            train_loader_list.append(mnist_train_loader)
            test_loader_list.append(mnist_test_loader)
            dset_train_list.append(mnist_dset_train)
            task_names.append(task)
            
        elif task == 'FMNIST':
            train_loader_list.append(fashion_mnist_train_loader)
            test_loader_list.append(fashion_mnist_test_loader)
            dset_train_list.append(fmnist_dset_train)
            task_names.append(task)

        elif task == 'KMNIST':
            train_loader_list.append(kmnist_train_loader)
            test_loader_list.append(kmnist_test_loader)
            dset_train_list.append(kmnist_dset_train)
            task_names.append(task)

        elif task == 'KaMNIST':
            train_loader_list.append(kamnist_train_loader)
            test_loader_list.append(kamnist_test_loader)
            dset_train_list.append(kamnist_dset_train)
            task_names.append(task)

    # Hyperparameters
    lr = args.lr
    epochs = args.epochs_per_task
    save_result = args.save
    #meta = args.meta
    ewc_lambda = args.ewc_lambda
    si_lambda = args.si_lambda
    archi = [args.in_size] + args.hidden_layers + [args.out_size]

    if args.net =='bnn':
        model = BNN( archi, init = args.init, width = args.init_width, norm = args.norm).to(device)
    elif args.net =='dnn':
        model = DNN( archi, init = args.init, width = args.init_width).to(device)
    elif args.net=='bcnn':
        model = ConvBNN(init = args.init, width = args.init_width, norm=args.norm).to(device)

    meta = {}
    for n, p in model.named_parameters():
        index = int(n[9])
        p.newname = 'l'+str(index)
        if ('fc' in n) or ('cv' in n):
            meta[p.newname] = args.meta[index-1] if len(args.meta)>1 else args.meta[0]



    print(model)
    #plot_parameters(model, path, save=save_result)

    previous_tasks_parameters = {}
    previous_tasks_fisher = {}

    # ewc parameters initialization
    if args.ewc:
        for n, p in model.named_parameters():
            if n.find('bn') == -1: #we dont store bn parameters as we allow task dependent bn
                n = n.replace('.', '__')
                previous_tasks_fisher[n] = []
                previous_tasks_parameters[n] = [] 
    elif args.si:
        W = {}
        p_prev = {}
        p_old = {}
        omega = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                n = n.replace('.', '__')
                W[n] = p.data.clone().zero_()
                omega[n] = p.data.clone().zero_()
                if args.net=='bnn':
                    p_prev[n] = p.data.clone()  # or sign
                    if args.bin_path:
                        p_old[n] = p.data.sign().clone()
                    else:
                        p_old[n] = p.data.clone()
                elif args.net=='dnn':
                    p_prev[n] = p.data.clone()
                    p_old[n] = p.data.clone()
    data = {}
    data['net'] = args.net
    data['scenario'] = args.scenario
    arch = ''
    if not(args.net=='bcnn'):
        for i in range(model.hidden_layers):
            arch = arch + '-' + str(model.layers_dims[i+1])

    data['arch'] = arch[1:]
    data['norm'] = args.norm
    data['lr'], data['meta'], data['ewc'], data['SI'], data['task_order'] = [], [], [], [], []  
    data['tsk'], data['epoch'], data['acc_tr'], data['loss_tr'] = [], [], [], []
    
    for i in range(len(test_loader_list)):
        data['acc_test_tsk_'+str(i+1)], data['loss_test_tsk_'+str(i+1)] = [], []

    name = '_'+data['net']+'_'+data['arch']+'_'
    for t in range(len(task_names)):
        if ('cifar100' in task_names[t]) and ('cifar100' in name):
            pass
        else:
            name = name+task_names[t]+'-'

    bn_states=[]        
    lrs = [lr*(args.gamma**(-i)) for i in range(len(train_loader_list))] 

    if args.beaker:
        optimizer = Adam_bk(model.parameters(), lr = lr, n_bk=args.n_bk, ratios=args.ratios, areas=args.areas, feedback=args.fb, meta=meta, weight_decay=args.decay, path=path)
    if args.si:
        optimizer = torch.optim.Adam(model.parameters(), lr = lr, weight_decay = args.decay)
    
    
    
    for task_idx, task in enumerate(train_loader_list):
        if not(args.beaker or args.si):
            optimizer = Adam_meta(model.parameters(), lr = lrs[task_idx], meta = meta, weight_decay = args.decay)

        for epoch in tqdm(range(1, epochs+1)):

            print('No.epoch:{}'.format((task_idx)*epochs+epoch))

            if args.ewc:
                train(model, task, task_idx, optimizer, device, args, prev_cons=previous_tasks_fisher, 
                        prev_params=previous_tasks_parameters) 
            elif args.si:
                train(model, task, task_idx, optimizer, device, args, prev_cons=omega, path_integ=W, prev_params=(p_prev, p_old) ) 
            else:
                train(model, task, task_idx, optimizer, device, args)

            data['task_order'].append(task_idx+1)
            data['tsk'].append(task_names[task_idx])
            data['epoch'].append(epoch)
            data['lr'].append(optimizer.param_groups[0]['lr'])

            train_accuracy, train_loss = test(model, args, task, device, verbose=True)

            data['acc_tr'].append(train_accuracy)
            data['loss_tr'].append(train_loss)
            data['meta'].append(meta)
            data['ewc'].append(ewc_lambda)
            data['SI'].append(si_lambda)

            current_bn_state = model.save_bn_states()

            for other_task_idx, other_task in enumerate(test_loader_list):

                print(other_task_idx)
                if args.scenario == 'task':
                    if other_task_idx>=task_idx:
                        model.load_bn_states(current_bn_state)
                        test_accuracy, test_loss = test(model ,args, other_task, device, verbose=(other_task_idx==task_idx))
                    else:
                        model.load_bn_states(bn_states[other_task_idx])
                        test_accuracy, test_loss = test(model , args, other_task, device)

                elif args.scenario =='domain':
                    test_accuracy, test_loss = test(model, args, other_task, device, verbose=True)

                data['acc_test_tsk_'+str(other_task_idx+1)].append(test_accuracy)
                print(data['acc_test_tsk_'+str(other_task_idx+1)])
                data['loss_test_tsk_'+str(other_task_idx+1)].append(test_loss)

            model.load_bn_states(current_bn_state)

        plot_parameters(model, path, save=save_result)
        # Uncomment for hidden weight histogram of Fig. 2g,h
        #time = datetime.now().strftime('%H-%M-%S')
        #for l in range(model.hidden_layers + 1):
        #    torch.save(model.layers['fc'+str(l+1)].weight.org.data, path+'/'+time+'_weights_fc'+str(l+1)+'.pt')

        #将已经test过的内容的BN参数进行保留
        bn_states.append(current_bn_state)

        #高级对比内容，可以忽略
        if args.ewc:
            fisher = estimate_fisher(model, dset_train_list[task_idx], device, num=5000, empirical=True)
            for n, p in model.named_parameters():
                if n.find('bn') == -1: # not batchnorm
                    n = n.replace('.', '__')

                    # random consolidation
                    if args.rnd_consolidation:
                        idx = torch.randperm(fisher[n].nelement())
                        previous_tasks_fisher[n].append(fisher[n].view(-1)[idx].view(fisher[n].size()))

                    # EWC consolidation, comment when using random consolidation
                    previous_tasks_fisher[n].append(fisher[n])
                    previous_tasks_parameters[n].append(p.detach().clone())

        elif args.si:
            omega = update_omega(model, omega, p_prev, W)
            for n, p in model.named_parameters():
                if n.find('bn') == -1: # not batchnorm
                    n = n.replace('.','__')
                    if args.net=='bnn':
                        p_prev[n] = p.org.detach().clone()  # or sign
                    else:
                        p_prev[n] = p.detach().clone()


    time = datetime.now().strftime('%H-%M-%S')
    df_data = pd.DataFrame(data)
    
    #保存相应的数据
    if save_result:
        
        #output_path = select_path + '\\' + 'dataset_sequence=_'+ str(s_num + 1)
        #if not(os.path.exists(output_path)):
        #    os.makedirs(output_path)
            
        df_data.to_csv(select_path +'\\'+'G_up_limit_70uS' + '.csv', index = False)
        


BNN(
  (layers): ModuleDict(
    (fc1): BinarizeLinear(in_features=784, out_features=1000, bias=False)
    (bn1): BatchNorm1d(1000, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (fc2): BinarizeLinear(in_features=1000, out_features=500, bias=False)
    (bn2): BatchNorm1d(500, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (fc3): BinarizeLinear(in_features=500, out_features=10, bias=False)
    (bn3): BatchNorm1d(10, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
)


  0%|          | 0/20 [00:00<?, ?it/s]

No.epoch:1


C:\Users\13170\AAA_BNN_CL\BNN_CL_simulation\models_utils.py:461: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha) (Triggered internally at  ..\torch\csrc\utils\python_arg_parser.cpp:1025.)
  grad.add_(group['weight_decay'], p.data)


Test accuracy: 57212/60000 (95.35%)
0
Test accuracy: 8505/10000 (85.05%)
[85.05]
1
[18.14]
2
[4.44]
3


...

Test accuracy: 10664/12000 (88.87%)
[10.81, 9.98, 9.67, 10.28, 10.17, 10.01, 10.21, 9.75, 9.77, 10.82, 10.08, 10.03, 9.03, 10.28, 10.03, 10.22, 9.84, 10.38, 10.45, 10.29, 9.38, 9.4, 11.06, 11.68, 11.97, 11.72, 11.97, 11.74, 12.77, 12.65, 13.22, 13.95, 12.05, 11.33, 11.53, 12.96, 11.26, 11.5, 11.92, 12.02, 17.37, 15.69, 17.66, 16.38, 17.5, 17.52, 17.7, 17.86, 16.27, 17.67, 17.49, 15.87, 17.96, 16.78, 18.61, 18.56, 17.68, 18.05, 19.12, 18.94, 80.45, 84.16, 86.13, 86.89, 87.79, 88.02, 88.27, 87.82, 88.68, 88.5, 88.47, 88.77, 88.67, 88.46, 88.63, 89.36, 88.77, 88.87]
No.epoch:79
Test accuracy: 43181/48000 (89.96%)
0
[85.05, 86.53, 87.05, 86.81, 86.71, 87.19, 87.02, 87.31, 87.12, 87.22, 86.6, 86.93, 87.02, 87.01, 87.25, 87.15, 86.93, 86.93, 87.04, 87.06, 86.98, 86.97, 86.91, 87.02, 86.99, 86.95, 87.02, 87.04, 87.01, 87.03, 86.98, 86.99, 87.07, 86.93, 87.03, 86.99, 87.07, 87.06, 87.08, 87.03, 87.03, 87.0, 87.05, 87.02, 86.98, 86.98, 87.02, 86.98, 87.02, 86.96, 87.03, 87.04, 86.96, 87.01, 86.

 95%|█████████▌| 19/20 [04:06<00:12, 12.98s/it]

Test accuracy: 10649/12000 (88.74%)
[10.81, 9.98, 9.67, 10.28, 10.17, 10.01, 10.21, 9.75, 9.77, 10.82, 10.08, 10.03, 9.03, 10.28, 10.03, 10.22, 9.84, 10.38, 10.45, 10.29, 9.38, 9.4, 11.06, 11.68, 11.97, 11.72, 11.97, 11.74, 12.77, 12.65, 13.22, 13.95, 12.05, 11.33, 11.53, 12.96, 11.26, 11.5, 11.92, 12.02, 17.37, 15.69, 17.66, 16.38, 17.5, 17.52, 17.7, 17.86, 16.27, 17.67, 17.49, 15.87, 17.96, 16.78, 18.61, 18.56, 17.68, 18.05, 19.12, 18.94, 80.45, 84.16, 86.13, 86.89, 87.79, 88.02, 88.27, 87.82, 88.68, 88.5, 88.47, 88.77, 88.67, 88.46, 88.63, 89.36, 88.77, 88.87, 88.74]
No.epoch:80
Test accuracy: 43166/48000 (89.93%)
0
[85.05, 86.53, 87.05, 86.81, 86.71, 87.19, 87.02, 87.31, 87.12, 87.22, 86.6, 86.93, 87.02, 87.01, 87.25, 87.15, 86.93, 86.93, 87.04, 87.06, 86.98, 86.97, 86.91, 87.02, 86.99, 86.95, 87.02, 87.04, 87.01, 87.03, 86.98, 86.99, 87.07, 86.93, 87.03, 86.99, 87.07, 87.06, 87.08, 87.03, 87.03, 87.0, 87.05, 87.02, 86.98, 86.98, 87.02, 86.98, 87.02, 86.96, 87.03, 87.04, 86.96, 87.

100%|██████████| 20/20 [04:19<00:00, 12.99s/it]

Test accuracy: 10688/12000 (89.07%)
[10.81, 9.98, 9.67, 10.28, 10.17, 10.01, 10.21, 9.75, 9.77, 10.82, 10.08, 10.03, 9.03, 10.28, 10.03, 10.22, 9.84, 10.38, 10.45, 10.29, 9.38, 9.4, 11.06, 11.68, 11.97, 11.72, 11.97, 11.74, 12.77, 12.65, 13.22, 13.95, 12.05, 11.33, 11.53, 12.96, 11.26, 11.5, 11.92, 12.02, 17.37, 15.69, 17.66, 16.38, 17.5, 17.52, 17.7, 17.86, 16.27, 17.67, 17.49, 15.87, 17.96, 16.78, 18.61, 18.56, 17.68, 18.05, 19.12, 18.94, 80.45, 84.16, 86.13, 86.89, 87.79, 88.02, 88.27, 87.82, 88.68, 88.5, 88.47, 88.77, 88.67, 88.46, 88.63, 89.36, 88.77, 88.87, 88.74, 89.07]


In [4]:
'''      
        #存储相应的权重mapping
        i=0
        for (n, p) in model.named_parameters():

            if (n.find('bias') == -1) and (len(p.size()) != 1):  #bias or batchnorm weight -> no plot
                if model.__class__.__name__.find('B') != -1:  #BVGG -> plot p.org
                    if hasattr(p,'org'):
                        weights_org = p.org.data.cpu().numpy()
                        weights = p.data.cpu().numpy()
                        np.savetxt(output_path+'\\'+'FC_'+str(i)+'_weights_org.csv',weights_org,delimiter=',')
                        np.savetxt(output_path+'\\'+'FC_'+str(i)+'_weights.csv',weights,delimiter=',')
                        i+=1
                    else:
                        pass
                else:
                    pass
            else:
                pass
                       
        #存储权重的flatten后的结果
        layer_num = 0
        for p in list(model.parameters()):
            dict_note={}
            if hasattr(p,'org'):
                #print(p.org)
                
                dict_note['org_weight']=p.org.flatten().cpu().numpy()
                dict_note['weight']=p.data.flatten().cpu().numpy()
                pd.DataFrame(dict_note['org_weight']).to_csv(output_path+'\\'+'FC_'+str(layer_num)+'_weight_org_flatten.csv')
                pd.DataFrame(dict_note['weight']).to_csv(output_path+'\\'+'FC_'+str(layer_num)+'_weight_flatten.csv')
                layer_num += 1
        
'''  

"      \n        #存储相应的权重mapping\n        i=0\n        for (n, p) in model.named_parameters():\n\n            if (n.find('bias') == -1) and (len(p.size()) != 1):  #bias or batchnorm weight -> no plot\n                if model.__class__.__name__.find('B') != -1:  #BVGG -> plot p.org\n                    if hasattr(p,'org'):\n                        weights_org = p.org.data.cpu().numpy()\n                        weights = p.data.cpu().numpy()\n                        np.savetxt(output_path+'\\'+'FC_'+str(i)+'_weights_org.csv',weights_org,delimiter=',')\n                        np.savetxt(output_path+'\\'+'FC_'+str(i)+'_weights.csv',weights,delimiter=',')\n                        i+=1\n                    else:\n                        pass\n                else:\n                    pass\n            else:\n                pass\n                       \n        #存储权重的flatten后的结果\n        layer_num = 0\n        for p in list(model.parameters()):\n            dict_note={}\n            if h